### Celda 1: Importación de librerías y configuración inicial


In [1]:
# =============================================================================
# PIPELINE DE INGESTA AUTOMATIZADA - ENDES 2025
# Curso: Lenguaje de Ciencia de Datos II
# =============================================================================

import os
import io
import zipfile
import logging
import urllib3
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configuración de logs
LOG_DIR = os.path.join("..", "logs")
os.makedirs(LOG_DIR, exist_ok=True) 

LOG_FILE = os.path.join(LOG_DIR, "ingesta_endes_2025.log")

# Limpiar handlers previos de Jupyter
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler()
    ]
)

# Deshabilitar advertencias SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuración de constantes
API_URL = "https://www.datosabiertos.gob.pe/api/3/action/package_show"
DATASET_ID = "4fad669c-a13d-4398-8d6e-c4e997faa75a"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Referer": "https://www.datosabiertos.gob.pe/"
}

# =============================================================================
# 2. CONFIGURACIÓN DE PARÁMETROS
# =============================================================================

# Directorio de salida para archivos Parquet (guardado dentro de data/)
OUTPUT_DIR = os.path.join("..", "data", "datos_normalizados_parquet")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Módulos específicos de ENDES 2025 (basados en el ZIP descargado)
MODULOS_A_PROCESAR = [
    '1036-Modulo1629', '1036-Modulo1630', '1036-Modulo1631', '1036-Modulo1632',
    '1036-Modulo1633', '1036-Modulo1634', '1036-Modulo1635', '1036-Modulo1636',
    '1036-Modulo1637', '1036-Modulo1638', '1036-Modulo1639', '1036-Modulo1640',
    '1036-Modulo1641'
]

print(f"   Configuración completada")
print(f"   Directorio de salida: {OUTPUT_DIR}")
print(f"   Módulos a procesar: {len(MODULOS_A_PROCESAR)}")

   Configuración completada
   Directorio de salida: ..\data\datos_normalizados_parquet
   Módulos a procesar: 13


### Celda 2: Funciones auxiliares


In [2]:
# =============================================================================
# FUNCIONES AUXILIARES
# =============================================================================

def detectar_formato_archivo(nombre_archivo):
    """
    Detecta el formato de un archivo basado en su extensión.
    """
    nombre_minuscula = nombre_archivo.lower()
    
    if nombre_minuscula.endswith('.sav'):
        return 'spss'
    elif nombre_minuscula.endswith('.dta'):
        return 'stata'
    elif nombre_minuscula.endswith('.csv') or nombre_minuscula.endswith('.txt'):
        return 'texto'
    else:
        return 'desconocido'


def leer_archivo_datos(archivo_bytes, formato):
    """
    Lee un archivo de datos según su formato.
    """
    if formato == 'spss':
        logging.info("   Leyendo archivo SPSS (.sav)...")
        return pd.read_spss(io.BytesIO(archivo_bytes))
    
    elif formato == 'stata':
        logging.info("   Leyendo archivo Stata (.dta)...")
        return pd.read_stata(io.BytesIO(archivo_bytes))
    
    elif formato == 'texto':
        logging.info("   Leyendo archivo de texto (.csv/.txt)...")
        try:
            df = pd.read_csv(
                io.BytesIO(archivo_bytes),
                sep=None,
                engine='python',
                encoding='latin-1',
                on_bad_lines='skip'
            )
            return df
        except:
            return pd.read_csv(
                io.BytesIO(archivo_bytes),
                sep=',',
                encoding='latin-1',
                on_bad_lines='skip'
            )
    
    else:
        raise ValueError(f"Formato no soportado: {formato}")


def normalizar_dataframe(df, nombre_modulo):
    """
    Aplica normalización estándar a un DataFrame.
    """
    logging.info(f"   Normalizando datos del módulo {nombre_modulo}...")
    
    # Limpiar nombres de columnas
    df.columns = df.columns.str.strip().str.lower()
    logging.info(f"   Columnas encontradas: {len(df.columns)}")
    
    # Normalizar strings
    columnas_objeto = df.select_dtypes(include=['object', 'category']).columns
    for col in columnas_objeto:
        try:
            if df[col].dtype.name == 'category':
                df[col] = df[col].cat.rename_categories(lambda x: str(x).strip() if pd.notna(x) else x)
            else:
                df[col] = df[col].str.strip() if df[col].dtype == 'object' else df[col]
        except Exception as e:
            logging.warning(f"   No se pudo normalizar columna {col}: {e}")
    
    # Eliminar filas completamente vacías
    df = df.dropna(how='all')
    
    # Convertir categorías a string (para compatibilidad con Parquet)
    for col in df.select_dtypes(include=['category']).columns:
        df[col] = df[col].astype(str)
    
    return df


def guardar_parquet(df, nombre_modulo):
    """
    Guarda un DataFrame en formato Parquet.
    """
    output_filepath = os.path.join(OUTPUT_DIR, f"{nombre_modulo}.parquet")
    
    try:
        df.to_parquet(
            output_filepath, 
            index=False, 
            engine='pyarrow',
            compression='snappy'
        )
        return output_filepath
    except ImportError:
        logging.warning("   pyarrow no instalado. Instalando...")
        import subprocess
        subprocess.check_call(['pip', 'install', 'pyarrow'])
        df.to_parquet(output_filepath, index=False, engine='pyarrow')
        return output_filepath

print("✅ Funciones auxiliares cargadas")

✅ Funciones auxiliares cargadas


### Celda 3: Función principal del pipeline


In [3]:
# =============================================================================
# FUNCIÓN PRINCIPAL DEL PIPELINE
# =============================================================================

def ejecutar_pipeline_ingesta():
    """
    Ejecuta el pipeline completo de ingesta de datos ENDES 2025.
    """
    logging.info("=" * 80)
    logging.info("=== INICIANDO PIPELINE DE INGESTA AUTOMATIZADA - ENDES 2025 ===")
    logging.info("=" * 80)
    
    # -------------------------------------------------------------------------
    # FASE 1: Consulta a la API
    # -------------------------------------------------------------------------
    logging.info(f"\n📡 FASE 1: Consultando API de Datos Abiertos")
    logging.info(f"   ID del dataset: {DATASET_ID}")
    
    try:
        parametros = {"id": DATASET_ID}
        response = requests.get(
            API_URL, 
            params=parametros, 
            headers=HEADERS, 
            verify=False,
            timeout=30
        )
        response.raise_for_status()
        datos_api = response.json()
        
        if not datos_api.get("success"):
            logging.error("❌ La API devolvió success=False.")
            return
        
        result_raw = datos_api.get("result")
        if isinstance(result_raw, list) and len(result_raw) > 0:
            result = result_raw[0]
        elif isinstance(result_raw, dict):
            result = result_raw
        else:
            logging.error("❌ Estructura 'result' incompatible.")
            return
        
        logging.info(f"   ✅ Dataset encontrado: {result.get('title')}")
        
        recursos = result.get("resources", [])
        logging.info(f"   📁 Recursos disponibles: {len(recursos)}")
        
    except Exception as e:
        logging.error(f"❌ Error en API: {e}")
        return
    
    # -------------------------------------------------------------------------
    # FASE 2: Identificar y descargar ZIP
    # -------------------------------------------------------------------------
    logging.info(f"\n📦 FASE 2: Identificando archivo ZIP")
    
    url_zip_principal = None
    for recurso in recursos:
        if recurso.get("format", "").lower() == "zip":
            url_zip_principal = recurso.get("url")
            break
    
    if not url_zip_principal:
        logging.error("❌ No se localizó el recurso ZIP.")
        return
    
    logging.info(f"   ✅ URL identificada: {url_zip_principal[:80]}...")
    
    try:
        logging.info(f"\n⬇️  FASE 3: Descargando ZIP...")
        descarga_response = requests.get(
            url_zip_principal, 
            headers=HEADERS, 
            verify=False, 
            stream=True,
            timeout=120
        )
        descarga_response.raise_for_status()
        
        zip_bytes_principal = io.BytesIO(descarga_response.content)
        tamaño_mb = len(descarga_response.content) / (1024 * 1024)
        logging.info(f"   ✅ Descarga completada: {tamaño_mb:.2f} MB")
        
    except Exception as e:
        logging.error(f"❌ Error al descargar: {e}")
        return
    
    # -------------------------------------------------------------------------
    # FASE 4: Procesar módulos
    # -------------------------------------------------------------------------
    logging.info(f"\n📂 FASE 4: Procesando módulos...")
    
    try:
        with zipfile.ZipFile(zip_bytes_principal) as zip_madre:
            archivos_zip_internos = zip_madre.namelist()
            logging.info(f"   Archivos en ZIP principal: {len(archivos_zip_internos)}")
            
            for index, modulo_buscado in enumerate(MODULOS_A_PROCESAR, 1):
                ruta_zip_interno = f"2025/{modulo_buscado}.zip"
                
                if ruta_zip_interno not in archivos_zip_internos:
                    logging.warning(f"[{index}/{len(MODULOS_A_PROCESAR)}] Módulo {modulo_buscado} no existe.")
                    continue
                    
                logging.info(f"[{index}/{len(MODULOS_A_PROCESAR)}] Procesando: {modulo_buscado}")
                
                try:
                    with zip_madre.open(ruta_zip_interno) as file_interno:
                        zip_bytes_interno = io.BytesIO(file_interno.read())
                        
                        with zipfile.ZipFile(zip_bytes_interno) as zip_hijo:
                            archivos_hijo = zip_hijo.namelist()
                            logging.info(f"   Contenido detectado: {archivos_hijo}")
                            
                            archivo_datos = None
                            formato_detectado = None
                            
                            for f in archivos_hijo:
                                formato = detectar_formato_archivo(f)
                                if formato != 'desconocido':
                                    archivo_datos = f
                                    formato_detectado = formato
                                    break
                            
                            if archivo_datos and formato_detectado:
                                logging.info(f"   ✅ Formato: {formato_detectado.upper()}")
                                
                                with zip_hijo.open(archivo_datos) as f_datos:
                                    raw_bytes = f_datos.read()
                                    df = leer_archivo_datos(raw_bytes, formato_detectado)
                                    
                                    if df is not None and not df.empty:
                                        df = normalizar_dataframe(df, modulo_buscado)
                                        ruta_parquet = guardar_parquet(df, modulo_buscado)
                                        logging.info(f"   ✅ Guardado: {ruta_parquet}")
                                        logging.info(f"      Filas: {len(df):,}, Columnas: {len(df.columns)}")
                                    else:
                                        logging.warning(f"   ⚠️ DataFrame vacío")
                            else:
                                logging.warning(f"   ⚠️ No se encontró archivo de datos")
                                
                except Exception as e_mod:
                    logging.error(f"   ❌ Error en {modulo_buscado}: {e_mod}")
                    
    except Exception as e:
        logging.error(f"❌ Error procesando ZIP: {e}")
        return
    
    # -------------------------------------------------------------------------
    # FASE 5: Resumen final
    # -------------------------------------------------------------------------
    logging.info("\n" + "=" * 80)
    logging.info("📊 RESUMEN FINAL")
    logging.info("=" * 80)
    
    archivos_generados = os.listdir(OUTPUT_DIR)
    archivos_parquet = [f for f in archivos_generados if f.endswith('.parquet')]
    
    logging.info(f"\n📁 Archivos Parquet generados: {len(archivos_parquet)}")
    
    for archivo in archivos_parquet:
        ruta = os.path.join(OUTPUT_DIR, archivo)
        tamaño_mb = os.path.getsize(ruta) / (1024 * 1024)
        try:
            df = pd.read_parquet(ruta)
            logging.info(f"   • {archivo}: {tamaño_mb:.2f} MB - {len(df):,} filas, {len(df.columns)} columnas")
        except:
            logging.info(f"   • {archivo}: {tamaño_mb:.2f} MB")
    
    logging.info("\n" + "=" * 80)
    logging.info("✅ PIPELINE FINALIZADO CON ÉXITO")
    logging.info("=" * 80)

print("✅ Función pipeline definida")

✅ Función pipeline definida


### Celda 4: Ejecutar el pipeline


In [4]:
# =============================================================================
# EJECUCIÓN DEL PIPELINE
# =============================================================================

# Ejecutar la ingesta completa
ejecutar_pipeline_ingesta()

# Listar archivos generados
archivos_creados = os.listdir(OUTPUT_DIR)
print("\n" + "=" * 60)
print("📁 ARCHIVOS PARQUET GENERADOS:")
print("=" * 60)
for archivo in sorted(archivos_creados):
    ruta = os.path.join(OUTPUT_DIR, archivo)
    tamaño = os.path.getsize(ruta) / (1024 * 1024)
    print(f"   • {archivo}: {tamaño:.2f} MB")
print("=" * 60)

2026-08-14 21:48:22,190 [INFO] ================================================================================
2026-08-14 21:48:22,192 [INFO] === INICIANDO PIPELINE DE INGESTA AUTOMATIZADA - ENDES 2025 ===
2026-08-14 21:48:22,193 [INFO] ================================================================================
2026-08-14 21:48:22,193 [INFO] 
📡 FASE 1: Consultando API de Datos Abiertos
2026-08-14 21:48:22,193 [INFO]    ID del dataset: 4fad669c-a13d-4398-8d6e-c4e997faa75a
2026-08-14 21:48:22,765 [INFO]    ✅ Dataset encontrado: Encuesta Demográfica y de Salud Familiar (ENDES) 2025 - [Instituto Nacional de Estadística e Informática - INEI]
2026-08-14 21:48:22,766 [INFO]    📁 Recursos disponibles: 7
2026-08-14 21:48:22,766 [INFO] 
📦 FASE 2: Identificando archivo ZIP
2026-08-14 21:48:22,770 [INFO]    ✅ URL identificada: https://www.datosabiertos.gob.pe/sites/default/files/2025.zip...
2026-08-14 21:48:22,771 [INFO] 
⬇️  FASE 3: Descargando ZIP...
2026-08-14 21:48:57,812 [INFO]    ✅ Des


📁 ARCHIVOS PARQUET GENERADOS:
   • 1036-Modulo1629.parquet: 0.80 MB
   • 1036-Modulo1630.parquet: 1.69 MB
   • 1036-Modulo1631.parquet: 1.80 MB
   • 1036-Modulo1632.parquet: 1.48 MB
   • 1036-Modulo1633.parquet: 0.69 MB
   • 1036-Modulo1634.parquet: 0.41 MB
   • 1036-Modulo1635.parquet: 1.21 MB
   • 1036-Modulo1636.parquet: 0.76 MB
   • 1036-Modulo1637.parquet: 0.71 MB
   • 1036-Modulo1638.parquet: 0.83 MB
   • 1036-Modulo1639.parquet: 0.18 MB
   • 1036-Modulo1640.parquet: 2.65 MB
   • 1036-Modulo1641.parquet: 0.39 MB


### Celda 5: Vista previa de los módulos


In [5]:
# =============================================================================
# VISTA PREVIA DE TODOS LOS MÓDULOS
# =============================================================================

print("\n" + "=" * 60)
print("📊 VISTA PREVIA DE MÓDULOS INGESTADOS")
print("=" * 60)

for modulo in MODULOS_A_PROCESAR:
    nombre_archivo = f"{modulo}.parquet"
    path_modulo = os.path.join(OUTPUT_DIR, nombre_archivo)
    
    print("\n" + "-" * 60)
    if os.path.exists(path_modulo):
        df_modulo = pd.read_parquet(path_modulo)
        print(f" Módulo: {modulo}")
        print(f" Dimensiones: {df_modulo.shape[0]:,} filas x {df_modulo.shape[1]} columnas")
        print(f" Columnas: {df_modulo.columns.tolist()[:5]}...")
        print(f"\n Primeras 3 filas:")
        display(df_modulo.head(3))
    else:
        print(f" ⚠️ Módulo {modulo}: No encontrado")


📊 VISTA PREVIA DE MÓDULOS INGESTADOS

------------------------------------------------------------
 Módulo: 1036-Modulo1629
 Dimensiones: 37,331 filas x 44 columnas
 Columnas: ['ï»¿id1', 'hhid', 'hv000', 'hv001', 'hv002']...

 Primeras 3 filas:


,ï»¿id1,hhid,hv000,hv001,hv002,hv002a,hv003,hv004,hv007,hv008,...,hv043,hv044,ubigeo,hv005,hv022,nconglome1,codccpp,nomccpp,longitudx,latitudy
0,2025,434400501,PE6,4344,5,1,2,4344,2025,1505,...,0,1,10101,73726,4,707201,1,CHACHAPOYAS,-77.868095,-6.216763
1,2025,434401201,PE6,4344,12,1,3,4344,2025,1505,...,0,1,10101,73726,4,707201,1,CHACHAPOYAS,-77.868095,-6.216763
2,2025,434402601,PE6,4344,26,1,1,4344,2025,1505,...,0,1,10101,73726,4,707201,1,CHACHAPOYAS,-77.868095,-6.216763



------------------------------------------------------------
 Módulo: 1036-Modulo1630
 Dimensiones: 37,331 filas x 130 columnas
 Columnas: ['ï»¿id1', 'hhid', 'hv201', 'hv202', 'hv204']...

 Primeras 3 filas:


,ï»¿id1,hhid,hv201,hv202,hv204,hv205,hv206,hv207,hv208,hv209,...,sh78,sh79,sh224,sh225u,sh225,sh227,qh227a,qh227b,hv270,hv271
0,2025,434400501,11,,996,11,1,0,1,0,...,0,,4,1,37,1,1,1,2,-.156810852906878
1,2025,434401201,11,,996,11,1,0,1,1,...,1,5,1,1,96,1,1,1,4,1.19692273142019
2,2025,434402601,11,,996,11,1,1,0,0,...,1,15,4,1,37,1,1,1,2,-.164400173142265



------------------------------------------------------------
 Módulo: 1036-Modulo1631
 Dimensiones: 36,260 filas x 103 columnas
 Columnas: ['ï»¿id1', 'caseid', 'hhid', 'v000', 'v001']...

 Primeras 3 filas:


...

,ï»¿id1,caseid,qcol93,q1035no,q1036n,q1037m,q1037p,q1037o,q1040a,q1040b,q1040c,q1040d,q1040e,q1040f,q1040g,q1040h,q1040i,q1040x
0,2025,434400501 2,1,1,AB,AB,B,,,,,,,,,,,
1,2025,434401201 1,1,1,Y,,,,,,,,,,,,,
2,2025,434403101 1,1,2,A,BE,,,1,1,1,,,,,,2,



------------------------------------------------------------
 Módulo: 1036-Modulo1640
 Dimensiones: 33,569 filas x 262 columnas
 Columnas: ['ï»¿id1', 'hhid', 'qhcluster', 'qhnumber', 'qhhome']...

 Primeras 3 filas:


,ï»¿id1,hhid,qhcluster,qhnumber,qhhome,qsnumero,qsintm,qsinty,qstotvisit,qsresult,...,qs901,qs902,qs903s,qs903d,qs905s,qs905d,qs906,qs907,qs908,peso15_amas
0,2025,434400501,4344,5,1,2,5,2025,1,1,...,154.5,4,102,55,94,55,1,83.6,1,158738.465868048
1,2025,434401201,4344,12,1,3,5,2025,1,1,...,153.6,1,132,64,133,64,1,100.2,1,198244.984986084
2,2025,434402601,4344,26,1,2,5,2025,2,1,...,148.4,4,94,60,96,64,1,61.5,1,194690.482396286



------------------------------------------------------------
 Módulo: 1036-Modulo1641
 Dimensiones: 33,569 filas x 21 columnas
 Columnas: ['ï»¿id1', 'hhid', 'qhcluster', 'qhnumber', 'qhhome']...

 Primeras 3 filas:


,ï»¿id1,hhid,qhcluster,qhnumber,qhhome,qh91,qh93,qh95,qh96,qh96a,...,qh96ac,qh97d,qh97m,qh97a,qh99,qh101,qh103,qh106,qhviolen,qh100b
0,2025,434400501,4344,5,1,,2,2,,,...,,,,,,2,2,2,2,2
1,2025,434401201,4344,12,1,,2,2,,,...,,,,,2,2,2,2,1,2
2,2025,434402601,4344,26,1,2,2,2,,,...,,,,,,2,2,2,2,2
